# Bài Thực Hành 2: Đóng gói Metadata & Chạy kiểm thử mô hình bằng Python

Sau khi đã tối ưu hóa mô hình ở bài thực hành số 1, bước tiếp theo trước khi deploy lên Android là **Nhúng Metadata** và **Kiểm thử mô hình** cục bộ bằng Python.

### Metadata là gì? Tại sao lại quan trọng?
Metadata (Siêu dữ liệu) là thông tin cấu trúc mô tả chi tiết mô hình được nhúng thẳng vào file `.tflite`. Nó cho ứng dụng Android biết:
- Tên mô hình, tác giả, mô tả.
- **Tham số tiền xử lý:** Cần chuẩn hóa màu bằng bao nhiêu (Mean, Std)? (ví dụ: `mean=127.5`, `std=127.5`).
- **File nhãn văn bản (.txt):** Nhúng trực tiếp file chứa các nhãn văn bản (ví dụ: lớp 0 là "chó", lớp 1 là "mèo") vào trong mô hình nhị phân nhị phân. Khi code Kotlin đọc mô hình, nó tự động trích xuất file nhãn này ra và map với mảng xác suất số thực.

Trong bài học này chúng ta sẽ:
1. Khám phá cách nhúng metadata bằng thư viện `tflite-support`.
2. Viết code Python chạy độc lập mô hình `.tflite` bằng `tf.lite.Interpreter` để xác thực độ chính xác.

### Bước 1: Cài đặt thư viện tflite-support
Hãy chạy lệnh cài đặt thư viện hỗ trợ xử lý metadata của Google:

In [ ]:
!pip install tflite-support

### Bước 2: Tạo file nhãn giả lập
Chúng ta cần một file `labels.txt` chứa danh sách tên các đối tượng phân loại. Hãy tạo một file mẫu giả lập có 3 nhãn phân loại:

In [ ]:
labels = ['background', 'chó', 'mèo']
with open('labels.txt', 'w', encoding='utf-8') as f:
    for label in labels:
        f.write(label + '\n')
print("Đã ghi file labels.txt thành công!")

### Bước 3: Nhúng Metadata bằng tflite-support
Đoạn code Python dưới đây khai báo cấu trúc metadata (tên model, preprocessing Mean/Std) và liên kết file `labels.txt` vào mô hình nhị phân của chúng ta.

In [ ]:
from tflite_support.metadata_writers import image_classifier
from tflite_support.metadata_writers import writer_utils

# 1. Khai báo các tham số cấu hình nhúng
model_path = "mobilenet_v2_dynamic_range.tflite" # Sử dụng model đã lưu ở Bài thực hành 1
save_to_path = "mobilenet_v2_with_metadata.tflite"
label_file = "labels.txt"

# 2. Cấu hình thông số chuẩn hóa ảnh (Mean = 127.5, Std = 127.5 để đưa pixel [0, 255] về dải thực [-1.0, 1.0])
input_mean = 127.5
input_std = 127.5

try:
    # 3. Khởi dựng bộ ghi metadata cho bài toán phân loại ảnh
    writer = image_classifier.MetadataWriter.create_for_inference(
        writer_utils.load_file(model_path),
        [input_mean],
        [input_std],
        [label_file]
    )

    # 4. Ghi đè thông tin metadata mô tả chung
    writer_metadata = writer.get_metadata_json()
    # print(writer_metadata) # Bạn có thể in ra chuỗi JSON mô tả cấu trúc

    # 5. Đóng gói và ghi ra file mô hình hoàn thiện mới
    writer_utils.save_file(writer.populate(), save_to_path)
    print("Đóng gói nhúng Metadata thành công! File mới đã sẵn sàng cho Android: ", save_to_path)
except Exception as e:
    print("Có lỗi xảy ra (Có thể do bạn chưa tạo file mô hình ở bài 1):", e)

### Bước 4: Chạy kiểm thử mô hình (Inference) bằng Python Interpreter độc lập
Trước khi viết code Android Studio, hãy luôn chạy mô hình trên Python PC để đảm bảo mô hình hoạt động chính xác và trích xuất đúng shape đầu ra.

In [ ]:
# 1. Khởi tạo đối tượng Interpreter từ file .tflite đã tối ưu
interpreter = tf.lite.Interpreter(model_path="mobilenet_v2_float32.tflite")
interpreter.allocate_tensors()

# 2. Lấy thông tin các cổng vào/ra (Tensors input/output)
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Cổng vào (Input Details):\n {input_details[0]}\n")
print(f"Cổng ra (Output Details):\n {output_details[0]}\n")

### Bước 5: Nạp dữ liệu giả lập và kích hoạt mô hình dự đoán
Chúng ta sẽ tạo một ảnh giả lập toàn màu xám (hoặc ngẫu nhiên) kích thước (1, 224, 224, 3) để chạy thử.

In [ ]:
# 1. Tạo một ảnh giả lập Float32
dummy_image = np.random.rand(1, 224, 224, 3).astype(np.float32)

# 2. Gán dữ liệu ảnh này vào cổng vào đầu tiên của mô hình
interpreter.set_tensor(input_details[0]['index'], dummy_image)

# 3. Kích hoạt tính toán (Inference)
interpreter.invoke()

# 4. Đọc kết quả mảng xác suất trả về từ cổng ra đầu ra
output_data = interpreter.get_tensor(output_details[0]['index'])
probabilities = output_data[0]

# Hiển thị top 5 nhãn có xác suất cao nhất
top_5_indices = np.argsort(probabilities)[-5:][::-1]
print("=" * 40)
print("Kết quả phân loại hàng đầu:")
print("=" * 40)
for idx in top_5_indices:
    print(f"Nhãn chỉ số {idx:<4} | Xác suất tự tin: {probabilities[idx]*100:>6.2f}%")
print("=" * 40)

### Kết luận:
Xin chúc mừng bạn! Bạn đã nắm trọn vẹn vòng đời tối ưu hóa của mô hình:
1. **Chuyển đổi** thành công mô hình học sâu lớn sang `.tflite` cực nhẹ.
2. **Tối ưu hóa lượng tử hóa** giúp giảm 4 lần dung lượng bộ nhớ.
3. **Đóng gói nhúng Metadata** và tệp nhãn nhãn.
4. **Chạy kiểm thử thành công** cục bộ qua bộ Interpreter Python thô.

Giờ đây, bạn có thể tự tin copy file `mobilenet_v2_with_metadata.tflite` này đưa trực tiếp vào thư mục `assets` của ứng dụng Android Studio và lập trình giao diện Camera thời gian thực thời gian thực!